In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

Using device: mps


In [5]:
model_name = "cross-encoder/msmarco-MiniLM-L6-en-de-v1"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
model.to(device)
model.eval()
print(f"Loaded model: {model_name}")
print(f"Number of labels: {model.config.num_labels}")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/msmarco-MiniLM-L6-en-de-v1
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded model: cross-encoder/msmarco-MiniLM-L6-en-de-v1
Number of labels: 1


In [6]:
dataset = load_dataset("glue", "mrpc", split="validation")
print("Dataset split: glue/mrpc validation")
print(f"Number of examples: {len(dataset)}")
print("Example row:")
print(dataset[0])

Dataset split: glue/mrpc validation
Number of examples: 408
Example row:
{'sentence1': "He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .", 'sentence2': '" The foodservice pie business does not fit our long-term growth strategy .', 'label': 1, 'idx': 9}


In [7]:
pair_lengths = []

for row in dataset:
    encoded = tokenizer(
        row["sentence1"],
        row["sentence2"],
        truncation=False,
        add_special_tokens=True
    )
    pair_lengths.append(len(encoded["input_ids"]))

sorted_lengths = sorted(pair_lengths)
n = len(sorted_lengths)
q1_threshold = sorted_lengths[n // 4]
q2_threshold = sorted_lengths[n // 2]
q3_threshold = sorted_lengths[(3 * n) // 4]

def assign_quartile(length):
    if length <= q1_threshold:
        return "q1_shortest"
    elif length <= q2_threshold:
        return "q2"
    elif length <= q3_threshold:
        return "q3"
    return "q4_longest"

quartiles = [assign_quartile(length) for length in pair_lengths]
quartile_names = ["q1_shortest", "q2", "q3", "q4_longest"]
quartile_counts = {name: quartiles.count(name) for name in quartile_names}

print(
    f"Length thresholds -> q1 <= {q1_threshold}, q2 <= {q2_threshold}, q3 <= {q3_threshold}, q4 > {q3_threshold}"
)
print("Quartile counts:")
print(quartile_counts)
print(f"Min length: {min(pair_lengths)}, Max length: {max(pair_lengths)}, Avg length: {sum(pair_lengths)/len(pair_lengths):.2f}")

Length thresholds -> q1 <= 52, q2 <= 64, q3 <= 75, q4 > 75
Quartile counts:
{'q1_shortest': 107, 'q2': 99, 'q3': 108, 'q4_longest': 94}
Min length: 28, Max length: 106, Avg length: 63.89


In [8]:
batch_size = 32
labels = dataset["label"]
predictions = []
confidences = []
positive_class_probs = []

for start_idx in range(0, len(dataset), batch_size):
    batch = dataset[start_idx:start_idx + batch_size]
    inputs = tokenizer(
        batch["sentence1"],
        batch["sentence2"],
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        if logits.shape[-1] == 1:
            pos_probs = torch.sigmoid(logits.squeeze(-1))
            probs = torch.stack([1.0 - pos_probs, pos_probs], dim=-1)
        else:
            probs = torch.softmax(logits, dim=-1)
            pos_probs = probs[:, 1]
        preds = (pos_probs >= 0.5).long()
    predictions.extend(preds.cpu().tolist())
    confidences.extend(torch.maximum(pos_probs, 1.0 - pos_probs).cpu().tolist())
    positive_class_probs.extend(pos_probs.cpu().tolist())

print(f"Completed inference for {len(predictions)} examples.")

Completed inference for 408 examples.


In [ ]:
accuracy = accuracy_score(labels, predictions)
precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average="binary", zero_division=0)
cm = confusion_matrix(labels, predictions)

print("Overall evaluation metrics:")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1       : {f1:.4f}")
print("Confusion matrix:")
print(cm)

In [ ]:
quartile_metrics = {}

for quartile_name in quartile_names:
    idxs = [i for i, q in enumerate(quartiles) if q == quartile_name]
    y_true = [labels[i] for i in idxs]
    y_pred = [predictions[i] for i in idxs]
    quartile_accuracy = accuracy_score(y_true, y_pred)
    quartile_precision, quartile_recall, quartile_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0
    )
    quartile_cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    quartile_metrics[quartile_name] = {
        "count": len(idxs),
        "accuracy": quartile_accuracy,
        "precision": quartile_precision,
        "recall": quartile_recall,
        "f1": quartile_f1,
        "confusion_matrix": quartile_cm.tolist()
    }

print("Quartile-wise metrics:")
for quartile_name in quartile_names:
    m = quartile_metrics[quartile_name]
    print(f"quartile={quartile_name} count={m['count']} accuracy={m['accuracy']:.4f} precision={m['precision']:.4f} recall={m['recall']:.4f} f1={m['f1']:.4f}")
    print(f"confusion_matrix={m['confusion_matrix']}")

In [ ]:
label_map = {0: "not_paraphrase", 1: "paraphrase"}
longest_quartile_mistakes = []

for i, quartile_name in enumerate(quartiles):
    if quartile_name == "q4_longest" and labels[i] != predictions[i]:
        longest_quartile_mistakes.append({
            "idx": i,
            "length": pair_lengths[i],
            "true_label": labels[i],
            "pred_label": predictions[i],
            "confidence": confidences[i],
            "positive_class_probability": positive_class_probs[i],
            "sentence1": dataset[i]["sentence1"],
            "sentence2": dataset[i]["sentence2"]
        })

longest_quartile_mistakes = sorted(longest_quartile_mistakes, key=lambda x: (-x["confidence"], -x["length"]))
num_examples_to_show = min(5, len(longest_quartile_mistakes))

print(f"Highest-confidence mistakes from longest quartile: showing {num_examples_to_show} of {len(longest_quartile_mistakes)}")
for item in longest_quartile_mistakes[:num_examples_to_show]:
    print(f"Index: {item['idx']}")
    print(f"Quartile: q4_longest | tokenized_pair_length: {item['length']}")
    print(f"sentence1: {item['sentence1']}")
    print(f"sentence2: {item['sentence2']}")
    print(f"true label: {item['true_label']} ({label_map[item['true_label']]})")
    print(f"pred label: {item['pred_label']} ({label_map[item['pred_label']]})")
    print(f"confidence: {item['confidence']:.4f}")
    print(f"positive_class_probability: {item['positive_class_probability']:.4f}")
    print("-" * 80)

In [ ]:
print("RESULT SUMMARY")
print(f"model={model_name}")
print("dataset_split=glue/mrpc validation")
print(f"device={device}")
print(f"num_examples={len(dataset)}")
print(f"overall_accuracy={accuracy:.4f}")
print(f"overall_precision={precision:.4f}")
print(f"overall_recall={recall:.4f}")
print(f"overall_f1={f1:.4f}")
print(f"length_threshold_q1={q1_threshold}")
print(f"length_threshold_q2={q2_threshold}")
print(f"length_threshold_q3={q3_threshold}")
for quartile_name in quartile_names:
    m = quartile_metrics[quartile_name]
    print(f"quartile_{quartile_name}_count={m['count']}")
    print(f"quartile_{quartile_name}_accuracy={m['accuracy']:.4f}")
    print(f"quartile_{quartile_name}_precision={m['precision']:.4f}")
    print(f"quartile_{quartile_name}_recall={m['recall']:.4f}")
    print(f"quartile_{quartile_name}_f1={m['f1']:.4f}")